In [1]:
#Use the following before running code
!python -m spacy download en_core_web_sm

zsh:1: command not found: python


In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from spacy import load
from csv import reader

def readFromCSV(path: str):
    data = None
    with open(path, 'r', encoding='UTF-8') as f:
        data = list(reader(f))
    return data

def preprocess(sents: list[str]):
    nlp = load("en_core_web_sm", disable=["ner", "parser"])
    out = []
    for sent in sents:
        sent = ' '.join(sent.split())
        doc = nlp(sent)
        out.append(' '.join(token.lemma_ for token in doc if not token.is_punct)) # keep stopwords as 'not' can sometimes be a stopword and ruin our sentiment analysis
    return out

def main():
    # Process labeled data
    labeledData = readFromCSV('data/labeledData.csv')[1:]
    unlabeledData = readFromCSV('data/unlabeledData.csv')[1:]

    labeledEmailMessages = preprocess([line[0] for line in labeledData])
    # labeledRejectionOutcomes = [0 if line[1] == 'reject' else 1 for line in labeledData]
    labeledRejectionOutcomes = [line[1] for line in labeledData]
    unlabeledEmailMessages = preprocess([line[9] for line in unlabeledData])
    #print(unlabeledEmailMessages)

    bowVectorizer = CountVectorizer(max_features=5000)

    X = bowVectorizer.fit_transform(labeledEmailMessages) #[line[0] for line in labeledData] is a list of strings
    y = labeledRejectionOutcomes

    # print(X.get_shape())
    # print(len(y))

    X_train, X_dev, y_train, y_dev = train_test_split(X, y)

    model = ComplementNB().fit(X_train, y_train)
   
    #print("Training F1")
    y_pred_dev = model.predict(X_dev)
    f1 = f1_score(y_dev, y_pred_dev, average='micro')
    print(f"F1 score: {f1:.4f}\n")

    #print("Test Results")
    X_test = bowVectorizer.transform(unlabeledEmailMessages)
    y_pred_test = model.predict(X_test)
    #print(*[reject + ' ' + message for message, reject in zip(unlabeledEmailMessages, y_pred_test)], sep='\n')

if __name__ == '__main__':
    main()


F1 score: 0.7879



In [4]:
# adding a 3rd and 4th status to labeledData, seeing as how some of the emails are not necessarily 'reject' or 'accept'
# Some of them are completely irrelevant, and some are just confirmations of an application, which is neither acceptance or rejection.
# kinda wondering if this was on purpose to train the model, but I don't know 

# Also I was researching making extensions for gmail, and it seems difficult, but if you see this before tuesday, I can always send the link to the discord
# (obviously I had to check the most reliable sources, reddit and stack overflow)

def main():
    # Process labeled data
    labeledData = [line for line in readFromCSV('temp-data/labeledData-extraStuff.csv')[1:] if line[1] != 'irrelevant'] 
    # not including the irrelevent emails in the training data for now?
    unlabeledData = readFromCSV('data/unlabeledData.csv')[1:]
    labeledEmailMessages = preprocess([line[0] for line in labeledData])
    labeledRejectionOutcomes = [line[1] for line in labeledData]
    unlabeledEmailMessages = preprocess([line[9] for line in unlabeledData])
    #print(unlabeledEmailMessages)

    bowVectorizer = CountVectorizer(max_features=5000)

    X = bowVectorizer.fit_transform(labeledEmailMessages) #[line[0] for line in labeledData] is a list of strings
    y = labeledRejectionOutcomes

    X_train, X_dev, y_train, y_dev = train_test_split(X, y)

    model = ComplementNB().fit(X_train, y_train)
   
    #print("Training F1")
    y_pred_dev = model.predict(X_dev)
    f1 = f1_score(y_dev, y_pred_dev, average='macro')
    print(f"F1 score: {f1:.4f}\n")

    if f1 == 1.0:
        print("Test results for perfect run")

    #print("Test Results")
        X_test = bowVectorizer.transform(unlabeledEmailMessages)
        y_pred_test = model.predict(X_test)
        print(*[reject + ': ' + message for message, reject in zip(unlabeledEmailMessages, y_pred_test)], sep='\n')

for i in range(50):
    if __name__ == '__main__':
        main()

F1 score: 0.7364

F1 score: 0.8857

F1 score: 1.0000

Test results for perfect run
reject: hello Micheal Gary Scott we ’ve receive your application for the Decision Scientist position at Tesco thank you for take the time to apply and for consider join we so what happen next first of all our recruitment team will take a look at your application if your skill and experience match the position we ’ll get in touch to arrange next step we know how much effort it take to put together an application so we review each one carefully as soon as we ’ve get any news for you we ’ll get in touch via email we be proud to have an inclusive culture at Tesco where everyone truly feel able to be themselves this be because we not only celebrate diversity but recognise the value and opportunity it bring therefore we be commit to create a workplace where difference be value and make sure that all colleague be give the same opportunity we ’re proud to have be accredit Disability Confident Leader and we ’re c

KeyboardInterrupt: 